In [4]:
import os
import math
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

TRAIN_PATH = 'dataset/public/train.csv'
TEST_PATH = 'dataset/public/test.csv'
OUTPUT_PATH = 'working/submission.csv'

MAX_CONTEXT_LEN = 512
CONTINUATION_LEN = 20
BATCH_SIZE = 128
EMBEDDING_DIM = 256
NUM_HEADS = 8
HIDDEN_DIM = 1024
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
EPOCHS = 20
MAX_LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
VOCAB = ['<PAD>', '<UNK>', '<BOS>'] + AMINO_ACIDS
CHAR_TO_IDX = {ch: idx for idx, ch in enumerate(VOCAB)}
IDX_TO_CHAR = {idx: ch for ch, idx in CHAR_TO_IDX.items()}
VOCAB_SIZE = len(VOCAB)
PAD_IDX = CHAR_TO_IDX['<PAD>']
BOS_IDX = CHAR_TO_IDX['<BOS>']

class ProteinDataset(Dataset):
    def __init__(self, df, is_test=False):
        self.df = df
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        context = row['context'][-MAX_CONTEXT_LEN:]
        src = [CHAR_TO_IDX.get(c, CHAR_TO_IDX['<UNK>']) for c in context]
        
        if self.is_test:
            return torch.tensor(src, dtype=torch.long), row['seq_id']
        else:
            continuation = row['continuation']
            tgt_full = [CHAR_TO_IDX.get(c, CHAR_TO_IDX['<UNK>']) for c in continuation]
            tgt_in = [BOS_IDX] + tgt_full[:-1]
            tgt_out = tgt_full
            return torch.tensor(src, dtype=torch.long), torch.tensor(tgt_in, dtype=torch.long), torch.tensor(tgt_out, dtype=torch.long)

def collate_train(batch):
    srcs, tgt_ins, tgt_outs = zip(*batch)
    srcs_pad = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgt_ins_pad = nn.utils.rnn.pad_sequence(tgt_ins, batch_first=True, padding_value=PAD_IDX)
    tgt_outs_pad = nn.utils.rnn.pad_sequence(tgt_outs, batch_first=True, padding_value=PAD_IDX)
    return srcs_pad, tgt_ins_pad, tgt_outs_pad

def collate_test(batch):
    srcs, seq_ids = zip(*batch)
    srcs_pad = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    return srcs_pad, seq_ids

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class ProteinSeq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.src_emb = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM, padding_idx=PAD_IDX)
        self.tgt_emb = nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(EMBEDDING_DIM)
        
        self.transformer = nn.Transformer(
            d_model=EMBEDDING_DIM,
            nhead=NUM_HEADS,
            num_encoder_layers=NUM_ENCODER_LAYERS,
            num_decoder_layers=NUM_DECODER_LAYERS,
            dim_feedforward=HIDDEN_DIM,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )
        self.fc_out = nn.Linear(EMBEDDING_DIM, VOCAB_SIZE)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        return mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0)).to(DEVICE)

    def forward(self, src, tgt, src_padding_mask, tgt_padding_mask):
        src_emb = self.pos_encoder(self.src_emb(src) * math.sqrt(EMBEDDING_DIM))
        tgt_emb = self.pos_encoder(self.tgt_emb(tgt) * math.sqrt(EMBEDDING_DIM))
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1))
        
        out = self.transformer(
            src=src_emb, tgt=tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask
        )
        return self.fc_out(out)

    def encode(self, src, src_padding_mask):
        src_emb = self.pos_encoder(self.src_emb(src) * math.sqrt(EMBEDDING_DIM))
        return self.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

    def decode(self, tgt, memory, memory_padding_mask):
        tgt_emb = self.pos_encoder(self.tgt_emb(tgt) * math.sqrt(EMBEDDING_DIM))
        tgt_mask = self.generate_square_subsequent_mask(tgt.size(1))
        return self.transformer.decoder(
            tgt_emb, memory, 
            tgt_mask=tgt_mask, 
            memory_key_padding_mask=memory_padding_mask
        )

def train_model():
    train_df = pd.read_csv(TRAIN_PATH)
    train_dataset = ProteinDataset(train_df, is_test=False)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_train)

    model = ProteinSeq2Seq().to(DEVICE)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=MAX_LR, steps_per_epoch=len(train_loader), epochs=EPOCHS, pct_start=0.1
    )

    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for src, tgt_in, tgt_out in progress_bar:
            src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
            
            src_padding_mask = (src == PAD_IDX)
            tgt_padding_mask = (tgt_in == PAD_IDX)
            
            optimizer.zero_grad()
            logits = model(src, tgt_in, src_padding_mask, tgt_padding_mask)
            
            loss = criterion(logits.view(-1, VOCAB_SIZE), tgt_out.view(-1))
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
            
    return model

def generate_predictions(model):
    test_df = pd.read_csv(TEST_PATH)
    test_dataset = ProteinDataset(test_df, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_test)

    model.eval()
    results = []

    with torch.no_grad():
        for src, seq_ids in tqdm(test_loader, desc="Generating Predictions"):
            src = src.to(DEVICE)
            batch_size = src.size(0)
            src_padding_mask = (src == PAD_IDX)
            
            memory = model.encode(src, src_padding_mask)
            tgt = torch.full((batch_size, 1), BOS_IDX, dtype=torch.long, device=DEVICE)
            
            for _ in range(CONTINUATION_LEN):
                out = model.decode(tgt, memory, src_padding_mask)
                logits = model.fc_out(out[:, -1, :])
                
                logits[:, PAD_IDX] = float('-inf')
                logits[:, BOS_IDX] = float('-inf')
                logits[:, CHAR_TO_IDX['<UNK>']] = float('-inf')
                
                next_token = logits.argmax(dim=-1).unsqueeze(1)
                tgt = torch.cat([tgt, next_token], dim=1)
                
            generated_sequences = tgt[:, 1:]
            
            for i in range(batch_size):
                tokens = generated_sequences[i].tolist()
                continuation = "".join([IDX_TO_CHAR.get(t, 'A') for t in tokens])
                results.append({
                    'seq_id': seq_ids[i],
                    'continuation': continuation
                })

    submission_df = pd.DataFrame(results)
    submission_df.to_csv(OUTPUT_PATH, index=False)
    print(f"Submission saved to {OUTPUT_PATH}")

if __name__ == "__main__":
    trained_model = train_model()
    generate_predictions(trained_model)

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(
Epoch 1/20:   0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
Generating Predictions: 100%|██████████| 4/4 [00:03<00:00,  1.16it/s]

Submission saved to /kaggle/working/submission.csv
